install pydantic, google-genai, instructor, httpx, beautifulsoup4, pypdf

In [70]:
import sys
from pathlib import Path
from dotenv import load_dotenv

notebook_dir = Path().resolve()

project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

sys.modules.pop("connectors", None)

load_dotenv(project_root / ".env", override=True)

print("The project root has been successfully added to the Python path:")
print(f"   -> {project_root}")


The project root has been successfully added to the Python path:
   -> D:\Cods\python_project\job-ai-assist


In [71]:
from local_connectors.resume_loader import load_resume_text

resume_path = project_root / "data" / "CV_OleksiiShcherbynin.pdf"
resume_text = load_resume_text(resume_path)

print("Resume text loaded successfully.")
print("Length of resume text:", len(resume_text))
print("First 500 characters of resume text:", resume_text[:500])


Resume text loaded successfully.
Length of resume text: 3539
First 500 characters of resume text: Oleksii Shcherbynin
Bratislava, Slovakia|[redacted-phone]|[redacted-email]|linkedin.com/in/oleksii-shcherbynin-692a5a410|
github.com/OleksiiShcherbynin
Education
Slovak University of Technology (STU)Bratislava, Slovakia
Bachelor of Science in Informatics (FIIT) Sep. 2025 – Present
• Relevant Coursework:Computer Programming (C, Java, Data Structures, Algorithms), Theoretical
Foundations of Computer Science (Discrete Mathematics, Mathematical Logic, Algebra), Computer Engineering
(Architect


In [72]:
import re

def senitize_text(text: str) -> str:
    text = text.replace('\uf0b7', '-')
    text = re.sub(r'\n{3,}', '\n\n ', text) # Replace 3 or more consecutive newlines with 2 newlines followed by a space
    text = re.sub(r' {2,}', ' ', text) # '  ' -> ' '

    return text.strip()

from local_connectors.resume_loader import load_resume_text
resume_path = project_root / "data" / "CV_OleksiiShcherbynin.pdf"
raw_text = load_resume_text(resume_path)

cleaned_text = senitize_text(raw_text)

print("Raw text length:", len(raw_text))
print("Cleaned text length:", len(cleaned_text))
print("First 500 characters of resume text:", cleaned_text[:500])

Raw text length: 3539
Cleaned text length: 3539
First 500 characters of resume text: Oleksii Shcherbynin
Bratislava, Slovakia|[redacted-phone]|[redacted-email]|linkedin.com/in/oleksii-shcherbynin-692a5a410|
github.com/OleksiiShcherbynin
Education
Slovak University of Technology (STU)Bratislava, Slovakia
Bachelor of Science in Informatics (FIIT) Sep. 2025 – Present
• Relevant Coursework:Computer Programming (C, Java, Data Structures, Algorithms), Theoretical
Foundations of Computer Science (Discrete Mathematics, Mathematical Logic, Algebra), Computer Engineering
(Architect


In [73]:
from core.models import (CandidateProfile, SearchPreferences, Vacancy, ExperienceEntry, Education, LanguageSkill, MatchResult)
from core.ports import VacancySource, MessageSender, MessageReceiver

print("Models and ports imported successfully.")

Models and ports imported successfully.


In [78]:
import os
import importlib
import local_connectors.llm as llm

importlib.reload(llm)
extract = llm.extract

from core.models import CandidateProfile

if not os.getenv("GOOGLE_API_KEY"):
    print("GOOGLE_API_KEY is not set, so extract cannot be called yet.")
else:
    profile = extract(
        resume_text,
        CandidateProfile,
        instruction=(
            "Extract the candidate's profile information from the resume text and populate the CandidateProfile data model. Ensure that all relevant fields are filled accurately based on the content of the resume."
            "If there is no field, leave None or an empty list."
        ),
    )

    print(f"👤 {profile.full_name}")
    print(f"   Roll:     {profile.title}")
    print(f"   Location:  {profile.location}")
    print(f"   Experience:   {profile.years_experience} years ({profile.seniority})")
    print(f"   Tech Stack:     {profile.tech_stack}")
    print(f"   Languages:     {[(l.language_code, l.level) for l in profile.languages]}")
    print(f"   Education:   {[(e.degree, e.field) for e in profile.education]}")
    print(f"\n📋 Resume:   {(profile.summary or '')[:150]}...")

👤 Oleksii Shcherbynin
   Roll:     Software Developer / Computer Science Student
   Location:  Bratislava, Slovakia
   Experience:   None years (Junior)
   Tech Stack:     ['Python', 'C (C99)', 'Java', 'SQL', 'LangChain', 'Qdrant Cloud', 'FastEmbed', 'RAG Architecture', 'NLP', 'Semantic Search', 'Streamlit', 'Git', 'Docker', 'JUnit 5', 'Gradle', 'GCC', 'REST APIs']
   Languages:     [('en', 'B1-B2'), ('sk', 'B2'), ('ru', 'Native'), ('uk', 'Native')]
   Education:   [('Bachelor of Science in Informatics (FIIT)', None), ('High School Diploma', None), ('Diploma in Programming and Software Development', None)]

📋 Resume:   Computer Science student and aspiring software developer with a strong foundation in algorithms, data structures, and AI/ML technologies. Proficient i...


In [81]:
from core.models import SearchPreferences, WorkFormat

prefs = SearchPreferences(
    desired_roles=["Software Engineer", "AI/ML Engineer", "Data Scientist"],
    min_salary=0,
    work_formats=[WorkFormat.ONSITE, WorkFormat.HYBRID],
    locations=["Bratislava", "Europe"],
    must_have=["student"],
    deal_breakers=["senior", "middle"],
)

print(f"Search Preferences: {prefs.desired_roles}, \nMin Salary: {prefs.min_salary}, \nWork Formats: {[wf.value for wf in prefs.work_formats]}, \nLocations: {prefs.locations}, \nMust Have: {prefs.must_have}, \nDeal Breakers: {prefs.deal_breakers}")

Search Preferences: ['Software Engineer', 'AI/ML Engineer', 'Data Scientist'], 
Min Salary: 0.0, 
Work Formats: ['onsite', 'hybrid'], 
Locations: ['Bratislava', 'Europe'], 
Must Have: ['student'], 
Deal Breakers: ['senior', 'middle']


In [87]:
from local_connectors.vacancy_source import ProfesiaSource

source = ProfesiaSource(use_cache=False, start_page=1, stop_page=5)

raw_postings = source.fetch_vacancies(limit=40)

print(f"Отримано оголошень: {len(raw_postings)}")
print("\n--- перше оголошення (URL + початок тексту) ---")
first = raw_postings[0]
print(first["url"])
print(first["title"])
print(first["text"][:600])

Отримано оголошень: 40

--- перше оголошення (URL + початок тексту) ---
https://www.profesia.sk/praca/bratislavsky-kraj/
Ponuky práce:Bratislavský kraj
Hľadanie práce Podľa regiónov Bratislavský kraj Banskobystrický kraj Žilinský kraj Trenčiansky kraj Trnavský kraj Nitriansky kraj Prešovský kraj Košický kraj Zahraničie Brigády Všetky ponuky Porovnanie mzdy Vytvorte si životopis Tip Prihlásiť S prihlásením je to lepšie Zistíte, či firma videla váš životopis, získate prehľad o histórii reakcií a na inzeráty odpoviete jednoduchšie s predvyplnenými údajmi. Rýchlo si tiež vytvoríte životopis a sprístupníte ho pre firmy. Prihlásiť sa Vytvorte si životopis Poradíme Vám? Vstup pre firmy S prihlásením je to lepšie Zistíte, či firma videla váš životopi


In [88]:
from local_connectors.llm import extract
from core.models import Vacancy

VACANCY_INSTRUCTION = (
    "Extract the structured data of the vacancy from the text of the announcement. "
    "Obviously, find out the URL of the vacancy (apply_url) - in the row 'URL: ...' on the cob."
    "Viznach language, which is written dumb (posting_language, ISO 639-1: 'sk', 'en'...). "
    "The raw_text field is left empty - please fill in the code."
    "If fields are missing — None or empty list."
)

vacancies: list[Vacancy] = []
for text in raw_postings:
    v = extract(text, Vacancy, instruction=VACANCY_INSTRUCTION)
    v.raw_text = text 
    vacancies.append(v)

print(f"Structured vacancies: {len(vacancies)}\n")
for v in vacancies:
    fmt = v.work_format.value if v.work_format else "—"
    lang = v.posting_language or "?"
    print(f"{(v.company or '—')[:20]:20} | {(v.role or '—')[:30]:30} | {fmt:6} | {lang}")

Structured vacancies: 40

—                    | —                              | —      | ?
—                    | —                              | —      | sk
—                    | —                              | —      | ?
—                    | Ponuky práce: Trenčiansky kraj | —      | sk
—                    | —                              | —      | sk
—                    | —                              | —      | ?
—                    | —                              | —      | sk
—                    | —                              | —      | sk
—                    | —                              | —      | sk
—                    | —                              | —      | sk
EURO-ŠTUKONZ, a.s.   | Majster stavebnej výroby       | onsite | sk
EURO-ŠTUKONZ, a.s.   | Stavbyvedúci - pozemné stavby  | —      | sk
Airvolute s.r.o.     | Výrobný pracovník – elektronik | hybrid | sk
All4Net s.r.o.       | Administratívny pracovník      | onsite | sk
TEBAU, spol. s r.o.  | Sk